In [ ]:
from google import genai
from google.genai import types
import os
import re

os.environ["GEMINI_API_KEY"] = "xxxxxx"

In [2]:
def extract_tag(text, tag):
    m = re.search(fr"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    return m.group(1).strip() if m else None


def parse_xml(text):
    return {
        "reason": extract_tag(text, "reason"),
        "img_search": extract_tag(text, "img_search"),
        "text_search": extract_tag(text, "text_search"),
        "answer": extract_tag(text, "answer")
    }

In [3]:
# init_prompt = """
# You are an expert visual assistant. Your task is to answer a user's question based on the provided image.

# **Step 1: Analyze the Image**
# Carefully examine the image and the user's question: {question}. Identify all recognizable entities, objects, text, and other visual clues.

# **Step 2: Plan Your Action**
# Based on your analysis, you must perform one of the following actions. You must include your thinking process inside a <reason>...</reason> block before choosing an action.

#     - **Action 1: Answer Directly**
#     If you can confidently identify the visual element and have the internal knowledge regarding the facts sufficient to answer the question, provide a direct, concise answer inside <answer>...</answer> tag.
#     Example: <answer>The construction of Eiffel Tower was finished on 03/31/1889.</answer>

#     - **Action 2: Use Image Search**
#     If you are not sure about the visual element and need to identify the visual element in the image, you can use one of the following image search methods.

#         - **Cropped search (Preferred for specific questions)**: Use this if the question is clearly about a specific visual element such as an object, person, animal, plant, aircraft, etc., or if the background is irrelevant. Describe the visual element concisely inside the <img_search>...</img_search> tags.
#         Example:
#         <img_search>the face of the person on the left</img_search>
#         <img_search>the red logo on the baseball cap</img_search>
    
#         - **Whole image search**: Only use this if the question is about the entire scene in general, its location, or the overall context. Output only: <img_search><img></img_search>.
#         Note: Do not output <img_search><img></img></img_search>.

#     - **Action 3: Use Text Search**
#     If you can identify the visual element confidently but need more specific information to answer the question, invoke the text search tool. Generate a focused query and output it as <text_search>your search query</text_search>.

# Remember, search results will be provided to you in subsequent turn. You can analyze the search results and decide your next action. You can perform image search only once, but have the option to perform multiple text searches to gather relevant information. All search results will be placed inside<information>...</information>.

# Here is the image and question:
# <image>
# {question}
# """.strip()

init_prompt = """
You are an expert visual assistant. Your task is to answer a user's question based on the provided image.

**Step 1: Analyze the Image**
Carefully examine the image and the user's question: {question}. Identify all recognizable entities, objects, text, and other visual clues.

**Step 2: Plan Your Action**
Based on your analysis, you must perform one of the following actions. You must include your thinking process inside a <reason>...</reason> block before choosing an action.

    - **Action 1: Use Image Search**
    If you are not sure about the visual element and need to identify the visual element in the image, you can use one of the following image search methods.

        - **Cropped search (Preferred for specific questions)**: Use this if the question is clearly about a specific visual element such as an object, person, animal, plant, aircraft, etc., or if the background is irrelevant. Describe the visual element concisely inside the <img_search>...</img_search> tags.
        Example:
        <img_search>the face of the person on the left</img_search>
        <img_search>the red logo on the baseball cap</img_search>
    
        - **Whole image search**: Only use this if the question is about the entire scene in general, its location, or the overall context. Output only: <img_search><img></img_search>.
        Note: Do not output <img_search><img></img></img_search>.

    - **Action 2: Use Text Search**
    If you can identify the visual element confidently but need more specific information to answer the question, invoke the text search tool. Generate a focused query and output it as <text_search>your search query</text_search>.

Remember, search results will be provided to you in subsequent turn. You can analyze the search results and decide your next action. You can perform image search only once, but have the option to perform multiple text searches to gather relevant information. All search results will be placed inside <information>...</information>.

Here is the image and question:
<image>
{question}
""".strip()

question = "What kind of wood is the bed made of?"

print(init_prompt.format(question=question))

You are an expert visual assistant. Your task is to answer a user's question based on the provided image.

**Step 1: Analyze the Image**
Carefully examine the image and the user's question: What kind of wood is the bed made of?. Identify all recognizable entities, objects, text, and other visual clues.

**Step 2: Plan Your Action**
Based on your analysis, you must perform one of the following actions. You must include your thinking process inside a <reason>...</reason> block before choosing an action.

    - **Action 1: Use Image Search**
    If you are not sure about the visual element and need to identify the visual element in the image, you can use one of the following image search methods.

        - **Cropped search (Preferred for specific questions)**: Use this if the question is clearly about a specific visual element such as an object, person, animal, plant, aircraft, etc., or if the background is irrelevant. Describe the visual element concisely inside the <img_search>...</img

In [ ]:
def run_image_search(query):
    pass

In [ ]:
def run_text_search(query):
    pass

In [ ]:
class DeepMMSearch:
    def __init__(self, api_key, model="gemini-2.5-pro"):
        if api_key is None:
            self.client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
        else:
            self.client = genai.Client(api_key=api_key)
        self.model = model
        self.history = []
    
    def reset(self):
        self.history = []
    
    @staticmethod
    def extract_tag(text, tag):
        m = re.search(fr"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
        return m.group(1).strip() if m else None
    
    def parse_xml(self, text):
        return {
            "reason": self.extract_tag(text, "reason"),
            "img_search": self.extract_tag(text, "img_search"),
            "text_search": self.extract_tag(text, "text_search"),
            "answer": self.extract_tag(text, "answer")
        }
    
    @staticmethod
    def load_image_bytes(image_path):
        with open(image_path, "rb") as f:
            image_bytes = f.read()
        return image_bytes
    
    def execute(self, question, image_path):
        image_bytes = self.load_image_bytes(image_path)
        self.history = [
            {"inline_data": {"mime_type": "image/jpeg", "data": image_bytes}},
            {"text": init_prompt.format(question=question)},
        ]
        
        used_img_search = False
        
        while True:
            response = self.client.models.generate_content(
                model=self.model,
                contents=self.history
            )
            xml = self.parse_xml(response.text)
            
            if xml["answer"]:
                return xml["answer"]

            if xml["img_search"]:
                if used_img_search:
                    self.history.append({"text": "<information>Image search already used.</information>"})
                else:
                    used_img_search = True
                    result = run_image_search(xml["img_search"])
                    # TODO: add GPT-4o based summarizer
                    self.history.append({"text": f"<information>{result}</information>"})
                continue

            if xml["text_search"]:
                result = run_text_search(xml["text_search"])
                # TODO: add GPT-4o based summarizer
                self.history.append({"text": f"<information>{result}</information>"})
                continue

            # No action, append error info to prevent deadlock
            self.history.append({"text": "<information>No valid action detected.</information>"})

        
        

In [14]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
chat  = client.chats.create(model="gemini-2.5-pro")

from PIL import Image

image = Image.open("test.jpg")

with open("test.jpg", "rb") as f:
    img_bytes = f.read()

response = chat.send_message(
    message=[image, init_prompt.format(question=question)]
)


# response = client.models.generate_content(
#     model="gemini-2.5-pro",
#     contents=[
#         {"text": init_prompt.format(question=question)},
#         {"inline_data": {"mime_type": "image/jpeg", "data": img_bytes}},
#     ],
# )
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""<reason>The user wants to identify the type of wood that the bed is made of. By examining the original image and the provided crops, I can observe the characteristics of the wood. The bed frame, as well as the matching wardrobe and bookshelf, is made of a light-colored, yellowish-brown wood. A key feature is the presence of prominent, darker knots and a distinct grain pattern. This combination of features is very characteristic of a particular type of wood commonly used for furniture. My initial assessment is that it is likely pine, specifically knotty pine. To confirm this and provide an accurate answer, I will use a text search to find information about woods with these visual properties.</reason>
<text_search>light colored wood with prominent knots used for furniture</text_search>"""
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishR

In [ ]:
result = parse_xml(response.text)

from pprint import pprint
pprint(result)

{'answer': None,
 'img_search': None,
 'reason': 'The user wants to identify the type of wood that the bed is made '
           'of. By examining the original image and the provided crops, I can '
           'observe the characteristics of the wood. The bed frame, as well as '
           'the matching wardrobe and bookshelf, is made of a light-colored, '
           'yellowish-brown wood. A key feature is the presence of prominent, '
           'darker knots and a distinct grain pattern. This combination of '
           'features is very characteristic of a particular type of wood '
           'commonly used for furniture. My initial assessment is that it is '
           'likely pine, specifically knotty pine. To confirm this and provide '
           'an accurate answer, I will use a text search to find information '
           'about woods with these visual properties.',
 'text_search': 'light colored wood with prominent knots used for furniture'}


In [18]:
history = chat.get_history()

In [19]:
history[0]

UserContent(
  parts=[
    Part(
      inline_data=Blob(
        data=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\t\t\x08\n\x0c\x14\r\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c $.\' ",#\x1c\x1c(7),01444\x1f\'9=82<.342\xff\xdb\x00C\x01\t\t\t...',
        mime_type='image/jpeg'
      )
    ),
    Part(
      text="""You are an expert visual assistant. Your task is to answer a user's question based on the provided image.

**Step 1: Analyze the Image**
Carefully examine the image and the user's question: What kind of wood is the bed made of?. Identify all recognizable entities, objects, text, and other visual clues.

**Step 2: Plan Your Action**
Based on your analysis, you must perform one of the following actions. You must include your thinking process inside a <reason>...</reason> block before choosing an action.

    - **Action 1: Use Image Search**
    If you are not sure about 

In [20]:
history[1]

Content(
  parts=[
    Part(
      text="""<reason>The user wants to identify the type of wood that the bed is made of. By examining the original image and the provided crops, I can observe the characteristics of the wood. The bed frame, as well as the matching wardrobe and bookshelf, is made of a light-colored, yellowish-brown wood. A key feature is the presence of prominent, darker knots and a distinct grain pattern. This combination of features is very characteristic of a particular type of wood commonly used for furniture. My initial assessment is that it is likely pine, specifically knotty pine. To confirm this and provide an accurate answer, I will use a text search to find information about woods with these visual properties.</reason>
<text_search>light colored wood with prominent knots used for furniture</text_search>"""
    ),
  ],
  role='model'
)